# ECL Deep Forecasting Benchmark — Dual-T4 Kaggle Notebook

This notebook prepares the raw UCI **ElectricityLoadDiagrams20112014 (ECL)** file and trains:

- **TimeMixer++**
- **TimePro**
- **xPatch**
- **PatchTST**
- **iTransformer**

It supports the paper-standard horizons **96, 192, 336, and 720** plus an operational **24-hour** horizon.  
An optional **168-hour** application horizon is included in configuration.

## What this notebook does

1. Converts the raw 370-client, 15-minute UCI file into the commonly used **321-client hourly benchmark**.
2. Uses a single leak-free chronological split and train-only standardization for every architecture.
3. Runs independent model/horizon jobs concurrently across Kaggle's two T4 GPUs.
4. Uses mixed precision, gradient clipping, early stopping, resumable checkpoints, and failure logs.
5. Reports:
   - normalized MSE, MAE, and RMSE for paper-style comparison;
   - physical-unit MAE, RMSE, sMAPE, MASE, bias;
   - **R² global** and **R² macro across clients**;
   - training and inference time.
6. Produces a final leaderboard CSV.

> **Important:** Published ECL papers normally compare normalized MSE/MAE. R² is added as a supplementary metric and should not replace MSE/MAE when comparing with paper tables.

## Research alignment and implementation sources

The notebook deliberately uses official repositories where available:

| Model | Source used here | Notes |
|---|---|---|
| TimePro | Official ICML 2025 repository | ECL defaults follow the repository script: `seq_len=96`, `d_model=32`, `e_layers=2`, `patch_len=8`, `stride=4`, batch size 16, and horizon-specific learning rates. |
| xPatch | Official AAAI 2025 repository | Uses the published decomposition + CNN/MLP architecture and RevIN. |
| iTransformer | THUML Time-Series-Library implementation | Same model family and standard long-term forecasting interface as the official repository. |
| PatchTST | THUML Time-Series-Library implementation | Uses patching, channel independence, and RevIN-compatible paper settings. |
| TimeMixer++ | PyPOTS public backbone implementation | The original paper did not provide a stable official public training repository when this notebook was authored. This implementation is clearly labeled rather than presented as an author-official reproduction. |

Primary references:

- TimeMixer++: https://arxiv.org/abs/2410.16032
- TimePro: https://arxiv.org/abs/2505.20774
- TimePro official code: https://github.com/xwmaxwma/TimePro
- xPatch: https://arxiv.org/abs/2412.17323
- xPatch official code: https://github.com/stitsyuk/xPatch
- iTransformer: https://arxiv.org/abs/2310.06625
- iTransformer official code: https://github.com/thuml/iTransformer
- PatchTST: https://arxiv.org/abs/2211.14730
- PatchTST official code: https://github.com/yuqinie98/PatchTST
- UCI ECL dataset DOI: https://doi.org/10.24432/C58C86

### Benchmark compatibility caveat

The raw UCI file contains **370 clients**. The classic multivariate benchmark removes 2011, retains **321 clients**, and converts the series to hourly consumption. This notebook recreates that transformation deterministically.

For strict, number-for-number reproduction of a specific paper table, use the exact `electricity.csv` distributed with that paper because seemingly minor differences in client filtering, timestamp handling, package versions, and random seeds can change results.

> **TimePro environment note:** its official repository pins PyTorch 2.0/CUDA 11.7 and requires compiled `selective_scan` and `DCNv4` operators. The notebook attempts to compile both against Kaggle's current image and isolates any failure so the remaining models still complete.

In [ ]:
# =========================
# 1. GLOBAL CONFIGURATION
# =========================
from pathlib import Path

RAW_ECL_PATH = Path(
    "/kaggle/input/datasets/jnesbit6/"
    "uc-irvine-time-series-electricity-dataset/LD2011_2014.txt"
)

WORK_ROOT = Path("/kaggle/working/ecl_deep_benchmark")
REPOS_DIR = WORK_ROOT / "repos"
DATA_DIR = WORK_ROOT / "data"
RUNS_DIR = WORK_ROOT / "runs"
LOGS_DIR = WORK_ROOT / "logs"

# Standard scientific horizons + essential day-ahead forecast.
HORIZONS = [24, 96, 192, 336, 720]

# Add this for the product-oriented weekly forecast:
OPTIONAL_APPLICATION_HORIZONS = [168]
INCLUDE_168H = False
if INCLUDE_168H:
    HORIZONS = sorted(set(HORIZONS + OPTIONAL_APPLICATION_HORIZONS))

MODELS = [
    "timemixerpp",
    "timepro",
    "xpatch",
    "patchtst",
    "itransformer",
]

SEQ_LEN = 96
LABEL_LEN = 48
N_CLIENTS = 321
SEED = 2026

# Paper scripts commonly use 10 epochs. Increase to 20–30 only after the
# full matrix works and the validation curves justify it.
MAX_EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 2
TRAIN_STRIDE = 1
EVAL_STRIDE = 1

# Set True for a quick end-to-end verification before the expensive run.
SMOKE_TEST = False

# When True, completed successful jobs are skipped.
RESUME = True

# Two independent jobs are scheduled concurrently, one on each T4.
MAX_PARALLEL_JOBS = 2

for p in [WORK_ROOT, REPOS_DIR, DATA_DIR, RUNS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Raw dataset:", RAW_ECL_PATH)
print("Work root:", WORK_ROOT)
print("Models:", MODELS)
print("Horizons:", HORIZONS)

In [ ]:
# ==========================================
# 2. INSTALL DEPENDENCIES AND CLONE SOURCES
# ==========================================
import os
import subprocess
import sys
from pathlib import Path

def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        check=check,
        text=True,
    )

# Kaggle Internet must be enabled for this first setup cell.
run_cmd([
    sys.executable, "-m", "pip", "install", "-q",
    "einops>=0.8",
    "timm>=1.0",
    "pypots>=1.5,<1.7",
    "reformer-pytorch",
    "local-attention",
    "rotary-embedding-torch",
    "scikit-learn>=1.4",
    "psutil",
    "ninja",
])

REPOSITORIES = {
    "timepro": (
        "https://github.com/xwmaxwma/TimePro.git",
        REPOS_DIR / "TimePro",
    ),
    "xpatch": (
        "https://github.com/stitsyuk/xPatch.git",
        REPOS_DIR / "xPatch",
    ),
    "tslib": (
        "https://github.com/thuml/Time-Series-Library.git",
        REPOS_DIR / "Time-Series-Library",
    ),
}

for name, (url, destination) in REPOSITORIES.items():
    if not destination.exists():
        run_cmd(["git", "clone", "--depth", "1", url, destination])
    else:
        print(f"{name}: already present at {destination}")


# TimePro imports the DCNv4 CUDA operator directly. Install it separately so
# a build failure does not prevent the other four models from running.
try:
    run_cmd([
        sys.executable, "-m", "pip", "install", "-q",
        "DCNv4==1.0.0.post2",
    ])
except subprocess.CalledProcessError:
    print(
        "\nWARNING: DCNv4 did not compile against the current Kaggle "
        "PyTorch/CUDA image. TimePro may fail, while the other models "
        "remain runnable. The official TimePro environment pins "
        "PyTorch 2.0 + CUDA 11.7."
    )

# TimePro uses CUDA extensions from its official repository.
# This may take several minutes the first time a Kaggle session starts.
timepro_ext = REPOS_DIR / "TimePro" / "selective_scan"
if timepro_ext.exists():
    marker = WORK_ROOT / ".timepro_extension_installed"
    if not marker.exists():
        try:
            run_cmd([
                sys.executable, "-m", "pip", "install", "-q", "-e", str(timepro_ext)
            ])
            marker.write_text("installed\n")
        except subprocess.CalledProcessError:
            print(
                "\nTimePro CUDA extension compilation failed. "
                "Other models can still run. Read the TimePro failure log later."
            )
    else:
        print("TimePro extension marker found; skipping recompilation.")
else:
    print("WARNING: TimePro/selective_scan was not found.")

print("\nRepository setup complete.")

In [ ]:
# ============================
# 3. VERIFY THE GPU ENVIRONMENT
# ============================
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"{props.total_memory / 1024**3:.1f} GiB | "
        f"compute {props.major}.{props.minor}"
    )

assert RAW_ECL_PATH.exists(), (
    f"Dataset not found at {RAW_ECL_PATH}. "
    "Attach the Kaggle dataset and verify the path."
)
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."

## Preprocessing policy

The raw values are power in kW at 15-minute intervals. Dividing each interval by four converts it to interval energy in kWh; summing four intervals produces hourly kWh.

The benchmark conversion below:

1. parses the Portuguese local timestamps;
2. drops 2011;
3. converts 15-minute power to hourly energy;
4. selects the 321 clients whose non-zero histories begin earliest;
5. keeps the first 26,211 hourly rows to match the widely used benchmark split lengths;
6. fits per-client mean and standard deviation on training data only.

The canonical split lengths used here are:

- train: 18,317 rows
- validation: 2,633 rows
- test: 5,261 rows

The scaler and transformed arrays are persisted so every model receives exactly the same data.

In [ ]:
# =====================================
# 4. PREPARE THE 321-CLIENT ECL BENCHMARK
# =====================================
import json
import numpy as np
import pandas as pd

STD_PATH = DATA_DIR / "ecl_321_standardized.npy"
RAW_HOURLY_PATH = DATA_DIR / "ecl_321_hourly_kwh.npy"
MARKS_PATH = DATA_DIR / "time_features.npy"
TIMESTAMPS_PATH = DATA_DIR / "timestamps.npy"
SCALER_PATH = DATA_DIR / "scaler.npz"
SPLIT_PATH = DATA_DIR / "split.json"
CLIENTS_PATH = DATA_DIR / "clients.json"
BENCHMARK_CSV_PATH = DATA_DIR / "electricity.csv"

TRAIN_LEN = 18_317
VAL_LEN = 2_633
TEST_LEN = 5_261
TOTAL_LEN = TRAIN_LEN + VAL_LEN + TEST_LEN

if all(
    p.exists()
    for p in [
        STD_PATH, RAW_HOURLY_PATH, MARKS_PATH, TIMESTAMPS_PATH,
        SCALER_PATH, SPLIT_PATH, CLIENTS_PATH
    ]
):
    print("Prepared benchmark files already exist; skipping raw conversion.")
else:
    print("Reading the 678 MB raw ECL file...")
    raw = pd.read_csv(
        RAW_ECL_PATH,
        sep=";",
        decimal=",",
        parse_dates=[0],
        index_col=0,
        low_memory=False,
    )
    raw.index = pd.DatetimeIndex(raw.index)
    raw = raw.sort_index()
    raw = raw.loc[raw.index >= pd.Timestamp("2012-01-01")]

    # UCI values are kW for each 15-minute interval.
    # Divide by 4 -> interval kWh; sum intervals -> hourly kWh.
    hourly = (
        raw.astype("float32")
        .div(4.0)
        .resample("1h")
        .sum()
        .astype("float32")
    )
    del raw

    # Deterministic approximation of the conventional 321-client subset:
    # retain clients that became active earliest after the 2011 removal.
    nonzero = hourly.to_numpy(copy=False) != 0
    has_activity = nonzero.any(axis=0)
    first_nonzero = np.where(
        has_activity,
        nonzero.argmax(axis=0),
        np.iinfo(np.int32).max,
    )
    order = np.argsort(first_nonzero, kind="stable")
    selected_idx = order[:N_CLIENTS]
    selected_clients = hourly.columns[selected_idx].tolist()
    hourly = hourly.iloc[:, selected_idx]

    assert hourly.shape[1] == N_CLIENTS
    assert len(hourly) >= TOTAL_LEN, (
        f"Need at least {TOTAL_LEN} hourly rows, got {len(hourly)}"
    )

    hourly = hourly.iloc[:TOTAL_LEN]
    timestamps = hourly.index.to_numpy(dtype="datetime64[ns]")
    raw_values = hourly.to_numpy(dtype=np.float32, copy=True)

    train_end = TRAIN_LEN
    val_end = TRAIN_LEN + VAL_LEN
    test_end = TOTAL_LEN

    train_values = raw_values[:train_end]
    mean = train_values.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = train_values.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(scale < 1e-6, 1.0, scale).astype(np.float32)
    standardized = ((raw_values - mean) / scale).astype(np.float32)

    dt = pd.DatetimeIndex(timestamps)
    # Continuous time features used by time-feature embeddings.
    marks = np.column_stack(
        [
            dt.month.to_numpy(dtype=np.float32) / 12.0 - 0.5,
            dt.day.to_numpy(dtype=np.float32) / 31.0 - 0.5,
            dt.dayofweek.to_numpy(dtype=np.float32) / 6.0 - 0.5,
            dt.hour.to_numpy(dtype=np.float32) / 23.0 - 0.5,
        ]
    ).astype(np.float32)

    np.save(STD_PATH, standardized)
    np.save(RAW_HOURLY_PATH, raw_values)
    np.save(MARKS_PATH, marks)
    np.save(TIMESTAMPS_PATH, timestamps)
    np.savez_compressed(SCALER_PATH, mean=mean, scale=scale)

    split = {
        "train_end": int(train_end),
        "val_end": int(val_end),
        "test_end": int(test_end),
        "train_len": int(TRAIN_LEN),
        "val_len": int(VAL_LEN),
        "test_len": int(TEST_LEN),
    }
    SPLIT_PATH.write_text(json.dumps(split, indent=2))
    CLIENTS_PATH.write_text(json.dumps(selected_clients, indent=2))

    # Standard custom-dataset format used by many forecasting repositories.
    export_df = pd.DataFrame(raw_values, columns=selected_clients)
    export_df.insert(0, "date", pd.DatetimeIndex(timestamps))
    export_df.to_csv(BENCHMARK_CSV_PATH, index=False)

    print("Prepared:", standardized.shape)
    print("Date range:", dt.min(), "to", dt.max())
    print("Saved benchmark CSV:", BENCHMARK_CSV_PATH)

split = json.loads(SPLIT_PATH.read_text())
scaler = np.load(SCALER_PATH)
print("Split:", split)
print("Mean shape:", scaler["mean"].shape)
print("Scale shape:", scaler["scale"].shape)

In [ ]:
# ========================================
# 5. DATA QUALITY AND LEAKAGE SANITY CHECKS
# ========================================
import json
import numpy as np
import pandas as pd

data_std = np.load(STD_PATH, mmap_mode="r")
data_raw = np.load(RAW_HOURLY_PATH, mmap_mode="r")
marks = np.load(MARKS_PATH, mmap_mode="r")
timestamps = np.load(TIMESTAMPS_PATH, mmap_mode="r")
split = json.loads(SPLIT_PATH.read_text())

assert data_std.shape == (split["test_end"], N_CLIENTS)
assert data_raw.shape == data_std.shape
assert marks.shape == (split["test_end"], 4)
assert np.isfinite(data_std).all()
assert np.isfinite(data_raw).all()

train_std = np.asarray(data_std[: split["train_end"]])
print("Train standardized mean, absolute average:", abs(train_std.mean(axis=0)).mean())
print("Train standardized std, average:", train_std.std(axis=0).mean())
print("Raw minimum:", float(data_raw.min()))
print("Raw maximum:", float(data_raw.max()))
print("Zero fraction:", float((data_raw == 0).mean()))
print("First timestamp:", pd.Timestamp(timestamps[0]))
print("Last timestamp:", pd.Timestamp(timestamps[-1]))

del train_std

## Why the notebook uses one process per experiment

These repositories use overlapping top-level Python package names such as `models` and `layers`. Importing all five into one interpreter can silently load the wrong module.

The generated worker script starts each experiment in a clean process with only the correct repository on `sys.path`. The scheduler assigns independent jobs to GPU 0 and GPU 1. This:

- avoids package-name collisions;
- uses both T4s;
- isolates CUDA-extension failures;
- allows resuming successful experiments;
- preserves one log and one checkpoint per model/horizon.

This is **multi-GPU experiment parallelism**, not data parallelism inside a single model. For a matrix of many model/horizon combinations, it is usually more efficient and more reliable on two 16-GB T4 GPUs.

In [ ]:
%%capture timepro_install

# =============================================
# BUILD AND VERIFY TIMEPRO CUDA EXTENSIONS
# =============================================

import importlib
import os
import subprocess
import sys
from pathlib import Path


TIMEPRO_DIR = Path(
    "/kaggle/working/ecl_deep_benchmark/repos/TimePro"
)
SELECTIVE_SCAN_DIR = TIMEPRO_DIR / "selective_scan"

assert TIMEPRO_DIR.exists(), (
    f"TimePro repository not found: {TIMEPRO_DIR}"
)

assert SELECTIVE_SCAN_DIR.exists(), (
    f"TimePro selective_scan directory not found: {SELECTIVE_SCAN_DIR}"
)


def run_quiet(command, *, cwd=None, env=None, description="Command"):
    """
    Run a subprocess without displaying its output.

    If the command fails, display only the last 40 output lines
    and raise an exception.
    """
    result = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.returncode != 0:
        output_lines = result.stdout.splitlines()
        error_tail = "\n".join(output_lines[-40:])

        raise RuntimeError(
            f"{description} failed with exit code "
            f"{result.returncode}.\n\n"
            f"Last output lines:\n{error_tail}"
        )

    return result


# Build tools only. This does not replace the existing PyTorch installation.
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "ninja",
        "packaging",
        "wheel",
        "setuptools",
    ],
    description="Build-tools installation",
)


build_env = os.environ.copy()

# Limit compilation parallelism to avoid exhausting Kaggle RAM.
build_env["MAX_JOBS"] = "2"

# Tesla T4 compute capability.
build_env["TORCH_CUDA_ARCH_LIST"] = "7.5"


# ---------------------------------------------
# Build TimePro selective_scan CUDA extension
# ---------------------------------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-build-isolation",
        "--no-deps",
        "--force-reinstall",
        ".",
    ],
    cwd=SELECTIVE_SCAN_DIR,
    env=build_env,
    description="TimePro selective-scan compilation",
)

importlib.invalidate_caches()

selective_scan = importlib.import_module(
    "selective_scan_cuda_oflex_rh"
)


# ---------------------------------------------
# Verify or install DCNv4
# ---------------------------------------------
try:
    dcnv4 = importlib.import_module("DCNv4")

except ModuleNotFoundError:
    run_quiet(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-build-isolation",
            "DCNv4",
        ],
        env=build_env,
        description="DCNv4 installation",
    )

    importlib.invalidate_caches()

    dcnv4 = importlib.import_module("DCNv4")


print("TimePro CUDA dependencies are ready.")
print("Selective scan:", selective_scan.__file__)
print("DCNv4:", dcnv4.__file__)

In [ ]:
import sys
from pathlib import Path

TIMEPRO_DIR = Path(
    "/kaggle/working/ecl_deep_benchmark/repos/TimePro"
)

if str(TIMEPRO_DIR) not in sys.path:
    sys.path.insert(0, str(TIMEPRO_DIR))

from model.TimePro import Model

print("TimePro import successful.")

In [ ]:
# ===================================
# 6. WRITE THE ISOLATED TRAINING WORKER
# ===================================
from pathlib import Path

WORKER_PATH = WORK_ROOT / "train_worker.py"
WORKER_SOURCE = '\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Dataset\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--root", type=Path, required=True)\n    p.add_argument("--model", type=str, required=True)\n    p.add_argument("--horizon", type=int, required=True)\n    p.add_argument("--gpu", type=int, required=True)\n    p.add_argument("--seq-len", type=int, default=96)\n    p.add_argument("--label-len", type=int, default=48)\n    p.add_argument("--epochs", type=int, default=10)\n    p.add_argument("--patience", type=int, default=3)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--train-stride", type=int, default=1)\n    p.add_argument("--eval-stride", type=int, default=1)\n    p.add_argument("--seed", type=int, default=2026)\n    p.add_argument("--smoke-test", action="store_true")\n    return p.parse_args()\n\n\nARGS = parse_args()\nos.environ["CUDA_VISIBLE_DEVICES"] = str(ARGS.gpu)\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nROOT = ARGS.root\nREPOS = ROOT / "repos"\nDATA = ROOT / "data"\nRUN_DIR = ROOT / "runs" / ARGS.model / f"h{ARGS.horizon}"\nRUN_DIR.mkdir(parents=True, exist_ok=True)\n\nMETRICS_PATH = RUN_DIR / "metrics.json"\nCHECKPOINT_PATH = RUN_DIR / "best.pt"\nHISTORY_PATH = RUN_DIR / "history.json"\nSAMPLE_PATH = RUN_DIR / "prediction_sample.npz"\n\n\ndef atomic_json(path: Path, payload: dict):\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(payload, indent=2, allow_nan=False))\n    tmp.replace(path)\n\n\ndef seed_everything(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\nseed_everything(ARGS.seed)\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\ntorch.backends.cudnn.benchmark = True\n\nDEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\nAMP_ENABLED = DEVICE.type == "cuda" and ARGS.model != "timemixerpp"\n\n\nclass WindowDataset(Dataset):\n    def __init__(\n        self,\n        values,\n        time_features,\n        region_start: int,\n        region_end: int,\n        seq_len: int,\n        pred_len: int,\n        stride: int = 1,\n        max_windows: int | None = None,\n    ):\n        first = int(region_start)\n        last_inclusive = int(region_end - seq_len - pred_len)\n        if last_inclusive < first:\n            raise ValueError(\n                f"No windows: start={first}, end={region_end}, "\n                f"seq={seq_len}, pred={pred_len}"\n            )\n        starts = np.arange(first, last_inclusive + 1, stride, dtype=np.int32)\n        if max_windows is not None:\n            starts = starts[:max_windows]\n        self.starts = starts\n        self.values = values\n        self.time_features = time_features\n        self.seq_len = seq_len\n        self.pred_len = pred_len\n\n    def __len__(self):\n        return len(self.starts)\n\n    def __getitem__(self, idx):\n        s = int(self.starts[idx])\n        e = s + self.seq_len\n        y_e = e + self.pred_len\n        x = np.asarray(self.values[s:e], dtype=np.float32)\n        y = np.asarray(self.values[e:y_e], dtype=np.float32)\n        x_mark = np.asarray(self.time_features[s:e], dtype=np.float32)\n        y_mark = np.asarray(self.time_features[e:y_e], dtype=np.float32)\n        return (\n            torch.from_numpy(x.copy()),\n            torch.from_numpy(y.copy()),\n            torch.from_numpy(x_mark.copy()),\n            torch.from_numpy(y_mark.copy()),\n        )\n\n\ndef make_loader(dataset, batch_size, shuffle):\n    kwargs = dict(\n        dataset=dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        num_workers=ARGS.workers,\n        pin_memory=True,\n        drop_last=shuffle,\n        persistent_workers=ARGS.workers > 0,\n    )\n    if ARGS.workers > 0:\n        kwargs["prefetch_factor"] = 2\n    return DataLoader(**kwargs)\n\n\ndef batch_size_for(model: str, horizon: int) -> int:\n    # Conservative defaults for 321 variables on a 16-GB T4.\n    if ARGS.smoke_test:\n        return 2\n    base = 16\n    if horizon >= 192:\n        base = 8\n    if horizon >= 336:\n        base = 4\n    if horizon >= 720:\n        base = 2\n    if model == "itransformer":\n        base = min(base, 8)\n    if model == "timemixerpp":\n        base = min(base, 4)\n    return max(1, base)\n\n\ndef learning_rate_for(model: str, horizon: int) -> float:\n    if model == "timepro":\n        return 5e-4 if horizon in (24, 96) else 3e-4\n    if model == "xpatch":\n        return 1e-4\n    if model == "itransformer":\n        return 1e-4\n    if model == "patchtst":\n        return 1e-4\n    if model == "timemixerpp":\n        return 1e-4\n    raise KeyError(model)\n\n\ndef clear_conflicting_modules():\n    # Each worker only imports one repository, but this protects notebook\n    # reruns and subprocess environments from stale top-level package names.\n    for name in list(sys.modules):\n        if name == "models" or name.startswith("models."):\n            del sys.modules[name]\n        if name == "layers" or name.startswith("layers."):\n            del sys.modules[name]\n\n\ndef build_model(name: str, pred_len: int, n_features: int):\n    clear_conflicting_modules()\n\n    if name == "timepro":\n        repo = REPOS / "TimePro"\n        sys.path.insert(0, str(repo))\n        from model.TimePro import Model\n\n        cfg = SimpleNamespace(\n            seq_len=ARGS.seq_len,\n            pred_len=pred_len,\n            use_norm=True,\n            patch_len=8,\n            stride=4,\n            d_model=32,\n            dropout=0.1,\n            e_layers=2,\n            enc_in=n_features,\n            dec_in=n_features,\n            c_out=n_features,\n        )\n        return Model(cfg), "encoder_decoder"\n\n    if name == "xpatch":\n        repo = REPOS / "xPatch"\n        sys.path.insert(0, str(repo))\n        from models.xPatch import Model\n\n        cfg = SimpleNamespace(\n            seq_len=ARGS.seq_len,\n            pred_len=pred_len,\n            enc_in=n_features,\n            patch_len=16,\n            stride=8,\n            padding_patch="end",\n            revin=True,\n            ma_type="ema",\n            alpha=0.3,\n            beta=0.1,\n        )\n        return Model(cfg), "x_only"\n\n    if name in {"patchtst", "itransformer"}:\n        repo = REPOS / "Time-Series-Library"\n        sys.path.insert(0, str(repo))\n\n        common = dict(\n            task_name="long_term_forecast",\n            seq_len=ARGS.seq_len,\n            label_len=ARGS.label_len,\n            pred_len=pred_len,\n            enc_in=n_features,\n            dec_in=n_features,\n            c_out=n_features,\n            output_attention=False,\n            embed="timeF",\n            freq="h",\n            activation="gelu",\n            dropout=0.1,\n            factor=3,\n            d_layers=1,\n            moving_avg=25,\n            distil=True,\n        )\n\n        if name == "itransformer":\n            from models.iTransformer import Model\n            cfg = SimpleNamespace(\n                **common,\n                d_model=512,\n                n_heads=8,\n                e_layers=4,\n                d_ff=512,\n                use_norm=True,\n                class_strategy="projection",\n            )\n            return Model(cfg), "encoder_decoder"\n\n        from models.PatchTST import Model\n        cfg = SimpleNamespace(\n            **common,\n            e_layers=3,\n            n_heads=16,\n            d_model=128,\n            d_ff=256,\n            fc_dropout=0.2,\n            head_dropout=0.0,\n            patch_len=16,\n            stride=8,\n            padding_patch="end",\n            revin=1,\n            affine=0,\n            subtract_last=0,\n            decomposition=0,\n            kernel_size=25,\n            individual=0,\n        )\n        return Model(cfg), "encoder_decoder"\n\n    if name == "timemixerpp":\n        # Public PyPOTS implementation of the architecture.\n        from pypots.nn.modules.timemixerpp.backbone import BackboneTimeMixerPP\n\n        model = BackboneTimeMixerPP(\n            task_name="long_term_forecast",\n            n_steps=ARGS.seq_len,\n            n_features=n_features,\n            n_pred_steps=pred_len,\n            n_pred_features=n_features,\n            n_layers=2,\n            d_model=32,\n            d_ffn=64,\n            n_heads=8,\n            dropout=0.1,\n            top_k=5,\n            n_kernels=6,\n            channel_mixing=True,\n            channel_independence=True,\n            downsampling_layers=2,\n            downsampling_window=2,\n            downsampling_method="avg",\n            use_future_temporal_feature=False,\n            use_norm=True,\n            embed="fixed",\n            freq="h",\n        )\n        return model, "timemixerpp"\n\n    raise KeyError(f"Unknown model: {name}")\n\n\ndef forward_model(model, interface, x, y, x_mark, y_mark):\n    if interface == "x_only":\n        return model(x)\n\n    if interface == "timemixerpp":\n        # PyPOTS RevIN interprets the second argument as an observation mask.\n        # ECL is complete, so None is correct and avoids treating calendar\n        # covariates as a missingness mask.\n        return model.forecast(x, None)\n\n    # The selected TimePro/iTransformer/PatchTST implementations are\n    # encoder-centric; a conventional decoder tensor keeps the interface\n    # compatible even when the model ignores it.\n    label = x[:, -ARGS.label_len :, :]\n    zeros = torch.zeros(\n        x.size(0),\n        ARGS.horizon,\n        x.size(2),\n        device=x.device,\n        dtype=x.dtype,\n    )\n    dec_inp = torch.cat([label, zeros], dim=1)\n    label_mark = x_mark[:, -ARGS.label_len :, :]\n    dec_mark = torch.cat([label_mark, y_mark], dim=1)\n    out = model(x, x_mark, dec_inp, dec_mark)\n    if isinstance(out, tuple):\n        out = out[0]\n    return out[:, -ARGS.horizon :, :]\n\n\ndef count_parameters(model):\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\n@torch.no_grad()\ndef normalized_mse(model, interface, loader):\n    model.eval()\n    squared_sum = 0.0\n    count = 0\n    for x, y, x_mark, y_mark in loader:\n        x = x.to(DEVICE, non_blocking=True)\n        y = y.to(DEVICE, non_blocking=True)\n        x_mark = x_mark.to(DEVICE, non_blocking=True)\n        y_mark = y_mark.to(DEVICE, non_blocking=True)\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ENABLED):\n            pred = forward_model(model, interface, x, y, x_mark, y_mark)\n        diff = pred.float() - y.float()\n        squared_sum += float(torch.sum(diff * diff).item())\n        count += diff.numel()\n    return squared_sum / max(count, 1)\n\n\nclass StreamingMetrics:\n    def __init__(self, n_features, mean, scale, mase_denominator):\n        self.n_features = n_features\n        self.mean = mean.astype(np.float64)\n        self.scale = scale.astype(np.float64)\n        self.mase_denom = np.maximum(mase_denominator.astype(np.float64), 1e-8)\n\n        self.n = 0\n        self.norm_abs = 0.0\n        self.norm_sq = 0.0\n\n        self.raw_abs = 0.0\n        self.raw_sq = 0.0\n        self.raw_signed = 0.0\n        self.smape_sum = 0.0\n        self.mase_sum = 0.0\n\n        self.y_sum = 0.0\n        self.y_sq_sum = 0.0\n        self.sse = 0.0\n\n        self.ch_count = np.zeros(n_features, dtype=np.int64)\n        self.ch_y_sum = np.zeros(n_features, dtype=np.float64)\n        self.ch_y_sq_sum = np.zeros(n_features, dtype=np.float64)\n        self.ch_sse = np.zeros(n_features, dtype=np.float64)\n\n    def update(self, pred_z, true_z):\n        pred_z = np.asarray(pred_z, dtype=np.float64)\n        true_z = np.asarray(true_z, dtype=np.float64)\n        diff_z = pred_z - true_z\n\n        self.norm_abs += np.abs(diff_z).sum()\n        self.norm_sq += np.square(diff_z).sum()\n\n        pred = pred_z * self.scale + self.mean\n        true = true_z * self.scale + self.mean\n        diff = pred - true\n\n        self.raw_abs += np.abs(diff).sum()\n        self.raw_sq += np.square(diff).sum()\n        self.raw_signed += diff.sum()\n        self.smape_sum += (\n            2.0 * np.abs(diff) / (np.abs(pred) + np.abs(true) + 1e-8)\n        ).sum()\n        self.mase_sum += (np.abs(diff) / self.mase_denom).sum()\n\n        self.n += true.size\n        self.y_sum += true.sum()\n        self.y_sq_sum += np.square(true).sum()\n        self.sse += np.square(diff).sum()\n\n        axes = tuple(range(true.ndim - 1))\n        points_per_channel = int(np.prod(true.shape[:-1]))\n        self.ch_count += points_per_channel\n        self.ch_y_sum += true.sum(axis=axes)\n        self.ch_y_sq_sum += np.square(true).sum(axis=axes)\n        self.ch_sse += np.square(diff).sum(axis=axes)\n\n    def compute(self):\n        n = max(self.n, 1)\n        global_sst = self.y_sq_sum - (self.y_sum ** 2) / n\n        r2_global = 1.0 - self.sse / max(global_sst, 1e-12)\n\n        ch_sst = self.ch_y_sq_sum - (\n            np.square(self.ch_y_sum) / np.maximum(self.ch_count, 1)\n        )\n        valid = ch_sst > 1e-12\n        ch_r2 = np.full(self.n_features, np.nan, dtype=np.float64)\n        ch_r2[valid] = 1.0 - self.ch_sse[valid] / ch_sst[valid]\n\n        return {\n            "normalized_mse": self.norm_sq / n,\n            "normalized_mae": self.norm_abs / n,\n            "normalized_rmse": math.sqrt(self.norm_sq / n),\n            "physical_mae_kwh": self.raw_abs / n,\n            "physical_rmse_kwh": math.sqrt(self.raw_sq / n),\n            "physical_bias_kwh": self.raw_signed / n,\n            "smape_percent": 100.0 * self.smape_sum / n,\n            "mase_24h": self.mase_sum / n,\n            "r2_global": float(r2_global),\n            "r2_macro_clients": float(np.nanmean(ch_r2)),\n            "r2_median_clients": float(np.nanmedian(ch_r2)),\n        }\n\n\n@torch.no_grad()\ndef evaluate(model, interface, loader, mean, scale, mase_denom):\n    model.eval()\n    meter = StreamingMetrics(\n        n_features=len(mean),\n        mean=mean,\n        scale=scale,\n        mase_denominator=mase_denom,\n    )\n    saved_pred = []\n    saved_true = []\n    start = time.perf_counter()\n\n    for x, y, x_mark, y_mark in loader:\n        x = x.to(DEVICE, non_blocking=True)\n        y = y.to(DEVICE, non_blocking=True)\n        x_mark = x_mark.to(DEVICE, non_blocking=True)\n        y_mark = y_mark.to(DEVICE, non_blocking=True)\n\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ENABLED):\n            pred = forward_model(model, interface, x, y, x_mark, y_mark)\n\n        pred_np = pred.float().cpu().numpy()\n        true_np = y.float().cpu().numpy()\n        meter.update(pred_np, true_np)\n\n        remaining = 16 - len(saved_pred)\n        if remaining > 0:\n            take = min(remaining, pred_np.shape[0])\n            saved_pred.extend(pred_np[:take])\n            saved_true.extend(true_np[:take])\n\n    elapsed = time.perf_counter() - start\n    metrics = meter.compute()\n    metrics["test_inference_seconds"] = elapsed\n    metrics["test_windows"] = len(loader.dataset)\n    metrics["seconds_per_window"] = elapsed / max(len(loader.dataset), 1)\n\n    if saved_pred:\n        np.savez_compressed(\n            SAMPLE_PATH,\n            pred_standardized=np.asarray(saved_pred, dtype=np.float32),\n            true_standardized=np.asarray(saved_true, dtype=np.float32),\n            mean=mean.astype(np.float32),\n            scale=scale.astype(np.float32),\n        )\n    return metrics\n\n\ndef main():\n    started = time.time()\n    split = json.loads((DATA / "split.json").read_text())\n    values = np.load(DATA / "ecl_321_standardized.npy", mmap_mode="r")\n    raw = np.load(DATA / "ecl_321_hourly_kwh.npy", mmap_mode="r")\n    marks = np.load(DATA / "time_features.npy", mmap_mode="r")\n    scaler = np.load(DATA / "scaler.npz")\n    mean = scaler["mean"]\n    scale = scaler["scale"]\n    n_features = values.shape[1]\n\n    train_end = split["train_end"]\n    val_end = split["val_end"]\n    test_end = split["test_end"]\n\n    # MASE seasonal denominator from training only.\n    seasonality = 24\n    train_raw = np.asarray(raw[:train_end], dtype=np.float64)\n    mase_denom = np.mean(\n        np.abs(train_raw[seasonality:] - train_raw[:-seasonality]),\n        axis=0,\n    )\n    del train_raw\n\n    max_windows = 64 if ARGS.smoke_test else None\n    train_ds = WindowDataset(\n        values, marks,\n        0, train_end,\n        ARGS.seq_len, ARGS.horizon,\n        stride=max(ARGS.train_stride, 16 if ARGS.smoke_test else ARGS.train_stride),\n        max_windows=max_windows,\n    )\n    val_ds = WindowDataset(\n        values, marks,\n        train_end - ARGS.seq_len, val_end,\n        ARGS.seq_len, ARGS.horizon,\n        stride=max(ARGS.eval_stride, 16 if ARGS.smoke_test else ARGS.eval_stride),\n        max_windows=max_windows,\n    )\n    test_ds = WindowDataset(\n        values, marks,\n        val_end - ARGS.seq_len, test_end,\n        ARGS.seq_len, ARGS.horizon,\n        stride=max(ARGS.eval_stride, 16 if ARGS.smoke_test else ARGS.eval_stride),\n        max_windows=max_windows,\n    )\n\n    batch_size = batch_size_for(ARGS.model, ARGS.horizon)\n    train_loader = make_loader(train_ds, batch_size, shuffle=True)\n    val_loader = make_loader(val_ds, batch_size, shuffle=False)\n    test_loader = make_loader(test_ds, batch_size, shuffle=False)\n\n    model, interface = build_model(ARGS.model, ARGS.horizon, n_features)\n    model = model.to(DEVICE)\n    params = count_parameters(model)\n    print(model)\n    print(f"Trainable parameters: {params:,}")\n    print(\n        f"Windows train/val/test: "\n        f"{len(train_ds)}/{len(val_ds)}/{len(test_ds)}"\n    )\n    print(f"Batch size: {batch_size}")\n\n    lr = learning_rate_for(ARGS.model, ARGS.horizon)\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=lr,\n        weight_decay=1e-4,\n    )\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n        optimizer,\n        T_max=max(ARGS.epochs, 1),\n        eta_min=lr * 0.05,\n    )\n    try:\n        scaler_amp = torch.amp.GradScaler(\n            "cuda", enabled=AMP_ENABLED\n        )\n    except (AttributeError, TypeError):\n        scaler_amp = torch.cuda.amp.GradScaler(\n            enabled=AMP_ENABLED\n        )\n    criterion = nn.MSELoss()\n\n    best_val = float("inf")\n    bad_epochs = 0\n    history = []\n    training_start = time.perf_counter()\n\n    for epoch in range(1, ARGS.epochs + 1):\n        model.train()\n        loss_sum = 0.0\n        element_count = 0\n        epoch_start = time.perf_counter()\n\n        for x, y, x_mark, y_mark in train_loader:\n            x = x.to(DEVICE, non_blocking=True)\n            y = y.to(DEVICE, non_blocking=True)\n            x_mark = x_mark.to(DEVICE, non_blocking=True)\n            y_mark = y_mark.to(DEVICE, non_blocking=True)\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\n                device_type="cuda",\n                dtype=torch.float16,\n                enabled=AMP_ENABLED,\n            ):\n                pred = forward_model(\n                    model, interface, x, y, x_mark, y_mark\n                )\n                loss = criterion(pred, y)\n\n            scaler_amp.scale(loss).backward()\n            scaler_amp.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler_amp.step(optimizer)\n            scaler_amp.update()\n\n            loss_sum += float(loss.item()) * y.numel()\n            element_count += y.numel()\n\n        scheduler.step()\n        train_mse = loss_sum / max(element_count, 1)\n        val_mse = normalized_mse(model, interface, val_loader)\n        epoch_seconds = time.perf_counter() - epoch_start\n\n        row = {\n            "epoch": epoch,\n            "train_mse": train_mse,\n            "val_mse": val_mse,\n            "lr": optimizer.param_groups[0]["lr"],\n            "seconds": epoch_seconds,\n        }\n        history.append(row)\n        atomic_json(HISTORY_PATH, history)\n        print(json.dumps(row))\n\n        if val_mse < best_val - 1e-7:\n            best_val = val_mse\n            bad_epochs = 0\n            torch.save(\n                {\n                    "model_state_dict": model.state_dict(),\n                    "model": ARGS.model,\n                    "horizon": ARGS.horizon,\n                    "seq_len": ARGS.seq_len,\n                    "n_features": n_features,\n                    "best_val_mse": best_val,\n                    "seed": ARGS.seed,\n                },\n                CHECKPOINT_PATH,\n            )\n        else:\n            bad_epochs += 1\n            if bad_epochs >= ARGS.patience:\n                print(f"Early stopping after epoch {epoch}")\n                break\n\n    training_seconds = time.perf_counter() - training_start\n\n    try:\n        checkpoint = torch.load(\n            CHECKPOINT_PATH, map_location=DEVICE, weights_only=False\n        )\n    except TypeError:\n        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    metrics = evaluate(\n        model,\n        interface,\n        test_loader,\n        mean=mean,\n        scale=scale,\n        mase_denom=mase_denom,\n    )\n    metrics.update(\n        {\n            "status": "success",\n            "model": ARGS.model,\n            "horizon": ARGS.horizon,\n            "seq_len": ARGS.seq_len,\n            "label_len": ARGS.label_len,\n            "seed": ARGS.seed,\n            "epochs_completed": len(history),\n            "best_val_normalized_mse": best_val,\n            "training_seconds": training_seconds,\n            "total_wall_seconds": time.time() - started,\n            "trainable_parameters": params,\n            "batch_size": batch_size,\n            "learning_rate": lr,\n            "gpu_visible": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",\n            "implementation_note": (\n                "PyPOTS public backbone"\n                if ARGS.model == "timemixerpp"\n                else "official/repository-aligned implementation"\n            ),\n        }\n    )\n    atomic_json(METRICS_PATH, metrics)\n    print("\\nFINAL_METRICS")\n    print(json.dumps(metrics, indent=2))\n\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except Exception as exc:\n        payload = {\n            "status": "failed",\n            "model": ARGS.model,\n            "horizon": ARGS.horizon,\n            "error_type": type(exc).__name__,\n            "error": str(exc),\n            "traceback": traceback.format_exc(),\n        }\n        atomic_json(METRICS_PATH, payload)\n        print(payload["traceback"], file=sys.stderr)\n        sys.exit(1)\n'
WORKER_PATH.write_text(WORKER_SOURCE)
print("Wrote:", WORKER_PATH)
print("Worker lines:", len(WORKER_SOURCE.splitlines()))

In [ ]:
# =======================================
# 7. STATICALLY VALIDATE THE WORKER SCRIPT
# =======================================
import ast
import py_compile

ast.parse(WORKER_PATH.read_text())
py_compile.compile(str(WORKER_PATH), doraise=True)
print("Worker syntax validation passed.")

## Recommended execution sequence

Running all models at all horizons is expensive. Use the following order:

1. Set `SMOKE_TEST = True` and run one or two representative jobs.
2. Set `SMOKE_TEST = False`.
3. Run the full standard benchmark.
4. Inspect validation curves and failures.
5. Tune only the top two models per horizon.
6. Repeat finalists with three seeds.

The scheduler below supports both smoke testing and the full matrix. It skips successful completed experiments when `RESUME=True`.

In [ ]:
# ==================================
# 8. BUILD THE EXPERIMENT JOB MATRIX
# ==================================
import itertools
import json
import sys
from pathlib import Path

def metrics_status(model, horizon):
    path = RUNS_DIR / model / f"h{horizon}" / "metrics.json"
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text()).get("status")
    except Exception:
        return "corrupt"

jobs = []
for model, horizon in itertools.product(MODELS, HORIZONS):
    if RESUME and metrics_status(model, horizon) == "success":
        print(f"SKIP completed: {model} h={horizon}")
        continue
    jobs.append({"model": model, "horizon": horizon})

if SMOKE_TEST:
    # One architecture from each source family, enough to validate interfaces.
    smoke_selection = {
        ("xpatch", 24),
        ("patchtst", 96),
        ("itransformer", 24),
        ("timemixerpp", 24),
        ("timepro", 24),
    }
    jobs = [
        job for job in jobs
        if (job["model"], job["horizon"]) in smoke_selection
    ]

print(f"Pending jobs: {len(jobs)}")
for job in jobs:
    print(job)

In [ ]:
# ======================================================
# 9. DUAL-T4 SCHEDULER — RUN TWO EXPERIMENTS CONCURRENTLY
# ======================================================
import os
import subprocess
import sys
import time
from collections import deque

assert torch.cuda.device_count() >= 2, (
    "This scheduler expects Kaggle's dual-T4 accelerator. "
    "Set MAX_PARALLEL_JOBS=1 and edit the assertion for a single GPU."
)

queue = deque(jobs)
available_gpus = deque(range(min(torch.cuda.device_count(), MAX_PARALLEL_JOBS)))
running = {}

def launch(job, gpu):
    model = job["model"]
    horizon = job["horizon"]
    log_path = LOGS_DIR / f"{model}_h{horizon}.log"
    cmd = [
        sys.executable,
        "-u",
        str(WORKER_PATH),
        "--root", str(WORK_ROOT),
        "--model", model,
        "--horizon", str(horizon),
        "--gpu", str(gpu),
        "--seq-len", str(SEQ_LEN),
        "--label-len", str(LABEL_LEN),
        "--epochs", str(MAX_EPOCHS),
        "--patience", str(PATIENCE),
        "--workers", str(NUM_WORKERS),
        "--train-stride", str(TRAIN_STRIDE),
        "--eval-stride", str(EVAL_STRIDE),
        "--seed", str(SEED),
    ]
    if SMOKE_TEST:
        cmd.append("--smoke-test")

    handle = open(log_path, "w", buffering=1)
    process = subprocess.Popen(
        cmd,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy(),
    )
    print(
        f"START pid={process.pid} gpu={gpu} "
        f"{model} horizon={horizon} log={log_path}"
    )
    return {
        "process": process,
        "handle": handle,
        "gpu": gpu,
        "job": job,
        "log": log_path,
        "started": time.time(),
    }

while queue or running:
    while queue and available_gpus:
        job = queue.popleft()
        gpu = available_gpus.popleft()
        info = launch(job, gpu)
        running[info["process"].pid] = info

    time.sleep(5)

    finished = []
    for pid, info in running.items():
        return_code = info["process"].poll()
        if return_code is None:
            continue
        info["handle"].close()
        elapsed = time.time() - info["started"]
        status = "OK" if return_code == 0 else f"FAILED({return_code})"
        print(
            f"END {status} gpu={info['gpu']} "
            f"{info['job']} elapsed={elapsed/60:.1f} min"
        )
        available_gpus.append(info["gpu"])
        finished.append(pid)

    for pid in finished:
        del running[pid]

print("All scheduled jobs finished.")

In [ ]:
# ====================================
# 10. BUILD AND DISPLAY THE LEADERBOARD
# ====================================
import json
from pathlib import Path

import pandas as pd

rows = []
failures = []

for model in MODELS:
    for horizon in HORIZONS:
        path = RUNS_DIR / model / f"h{horizon}" / "metrics.json"
        if not path.exists():
            continue
        payload = json.loads(path.read_text())
        if payload.get("status") == "success":
            rows.append(payload)
        else:
            failures.append(payload)

leaderboard = pd.DataFrame(rows)
if not leaderboard.empty:
    preferred_columns = [
        "horizon",
        "model",
        "normalized_mse",
        "normalized_mae",
        "normalized_rmse",
        "r2_global",
        "r2_macro_clients",
        "physical_mae_kwh",
        "physical_rmse_kwh",
        "smape_percent",
        "mase_24h",
        "physical_bias_kwh",
        "best_val_normalized_mse",
        "trainable_parameters",
        "epochs_completed",
        "training_seconds",
        "test_inference_seconds",
        "batch_size",
        "seed",
    ]
    existing = [c for c in preferred_columns if c in leaderboard.columns]
    leaderboard = (
        leaderboard[existing]
        .sort_values(["horizon", "normalized_mse"])
        .reset_index(drop=True)
    )
    leaderboard.to_csv(WORK_ROOT / "leaderboard.csv", index=False)
    display(leaderboard)
else:
    print("No successful metrics found yet.")

if failures:
    print("\nFailures:")
    failure_df = pd.DataFrame(failures)
    display(
        failure_df[
            [c for c in ["model", "horizon", "error_type", "error"] if c in failure_df]
        ]
    )

In [ ]:
# ==========================================
# 11. BEST MODEL PER HORIZON AND PAPER METRICS
# ==========================================
if not leaderboard.empty:
    winners = (
        leaderboard.sort_values("normalized_mse")
        .groupby("horizon", as_index=False)
        .first()
    )
    display(
        winners[
            [
                "horizon",
                "model",
                "normalized_mse",
                "normalized_mae",
                "r2_global",
                "r2_macro_clients",
                "physical_mae_kwh",
                "smape_percent",
            ]
        ]
    )

    paper_table = leaderboard[
        leaderboard["horizon"].isin([96, 192, 336, 720])
    ][
        [
            "model",
            "horizon",
            "normalized_mse",
            "normalized_mae",
            "r2_global",
        ]
    ].copy()

    paper_table.to_csv(WORK_ROOT / "paper_comparison_metrics.csv", index=False)
    display(paper_table)

In [ ]:
# ===========================================
# 12. PLOT VALIDATION CURVES FOR ONE EXPERIMENT
# ===========================================
import json
import matplotlib.pyplot as plt
import pandas as pd

MODEL_TO_PLOT = "xpatch"
HORIZON_TO_PLOT = 96

history_path = (
    RUNS_DIR / MODEL_TO_PLOT / f"h{HORIZON_TO_PLOT}" / "history.json"
)

if history_path.exists():
    history = pd.DataFrame(json.loads(history_path.read_text()))

    plt.figure(figsize=(8, 4))
    plt.plot(history["epoch"], history["train_mse"], marker="o", label="train")
    plt.plot(history["epoch"], history["val_mse"], marker="o", label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Normalized MSE")
    plt.title(f"{MODEL_TO_PLOT} — horizon {HORIZON_TO_PLOT}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("History not found:", history_path)

In [ ]:
# ====================================
# 13. INSPECT A SAVED FORECAST EXAMPLE
# ====================================
import matplotlib.pyplot as plt
import numpy as np

MODEL_TO_INSPECT = "xpatch"
HORIZON_TO_INSPECT = 96
CLIENT_INDEX = 0
SAMPLE_INDEX = 0

sample_path = (
    RUNS_DIR / MODEL_TO_INSPECT
    / f"h{HORIZON_TO_INSPECT}"
    / "prediction_sample.npz"
)

if sample_path.exists():
    sample = np.load(sample_path)
    pred_z = sample["pred_standardized"][SAMPLE_INDEX, :, CLIENT_INDEX]
    true_z = sample["true_standardized"][SAMPLE_INDEX, :, CLIENT_INDEX]
    mean = sample["mean"][CLIENT_INDEX]
    scale = sample["scale"][CLIENT_INDEX]

    pred = pred_z * scale + mean
    true = true_z * scale + mean

    plt.figure(figsize=(11, 4))
    plt.plot(true, label="actual")
    plt.plot(pred, label="forecast")
    plt.xlabel("Forecast hour")
    plt.ylabel("Hourly energy (kWh)")
    plt.title(
        f"{MODEL_TO_INSPECT} | horizon={HORIZON_TO_INSPECT} "
        f"| client={CLIENT_INDEX}"
    )
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Sample not found:", sample_path)

## How to interpret the metrics

### Metrics used for paper comparison

- **Normalized MSE:** primary optimization/evaluation metric in most long-term forecasting tables.
- **Normalized MAE:** second standard paper metric.
- **Normalized RMSE:** square root of normalized MSE.

### Application metrics

- **Physical MAE/RMSE:** reported in hourly kWh after inverse transformation.
- **sMAPE:** scale-independent percentage error, but can still be unstable around zero.
- **MASE-24h:** MAE relative to a daily seasonal-naive denominator computed only from training data.
- **Bias:** indicates systematic over- or under-prediction.

### R²

- **R² global:** flattens clients, forecast origins, and horizons into one population.
- **R² macro clients:** calculates R² per client and averages across valid clients.
- **R² median clients:** robust indication of the typical client.

R² can be negative. A negative value means that, under that aggregation, the forecasts are worse than predicting the corresponding mean. Because high-load clients can dominate global R², always report macro/median client R² beside it.

## Recommended scientific follow-up after the first successful run

Do not immediately tune all 25+ experiments.

1. Verify that standard-horizon MSE/MAE are in the same general range as published results.
2. Select the top two models at each horizon.
3. Tune only those finalists:
   - learning rate;
   - dropout;
   - model width/depth;
   - patch length and stride;
   - lookback length;
   - decomposition parameters for xPatch.
4. Repeat each finalist with seeds such as `2026`, `2027`, and `2028`.
5. Report mean ± standard deviation.
6. Keep the final test partition untouched during tuning.
7. Retrain the selected configuration on train + validation only after model selection, then perform one final test evaluation.
8. Export one artifact for each operational horizon: 24h, 168h, and 720h.

### Important TimeMixer++ limitation

This notebook uses a public PyPOTS implementation because an author-maintained official training repository was not available in the sources reviewed. Treat its numbers as an implementation-based experiment, not an exact official-code reproduction. TimePro and xPatch are closer to direct official-code reproduction.

## Output files

After training, the notebook writes:

```text
/kaggle/working/ecl_deep_benchmark/
├── data/
│   ├── electricity.csv
│   ├── ecl_321_hourly_kwh.npy
│   ├── ecl_321_standardized.npy
│   ├── time_features.npy
│   ├── timestamps.npy
│   ├── scaler.npz
│   └── split.json
├── runs/
│   └── <model>/h<horizon>/
│       ├── best.pt
│       ├── history.json
│       ├── metrics.json
│       └── prediction_sample.npz
├── logs/
│   └── <model>_h<horizon>.log
├── leaderboard.csv
├── paper_comparison_metrics.csv
└── train_worker.py
```

Download the entire `ecl_deep_benchmark` directory or package it as a Kaggle output dataset so checkpoints survive beyond the notebook session.